Data import: exploration notebook

In [5]:
import requests
import pandas as pd
import math
from pathlib import Path
import logging

Indicators come from the World Bank website:
- Population: https://data.worldbank.org/indicator/SP.POP.TOTL
- Surface area: https://data.worldbank.org/indicator/AG.SRF.TOTL.K2
- GDP per capita:
    - PPP: https://data.worldbank.org/indicator/NY.GDP.PCAP.PP.CD?most_recent_year_desc=true&year=2024
    - Current USD: https://data.worldbank.org/indicator/NY.GDP.PCAP.CD?most_recent_year_desc=true&year=2024
- Life expectancy: https://data.worldbank.org/indicator/SP.DYN.LE00.IN
- CO2 per capita: https://data.worldbank.org/indicator/EN.GHG.CO2.PC.CE.AR5
- PM2.5: https://data.worldbank.org/indicator/EN.ATM.PM25.MC.M3
- Urban population (%): https://data.worldbank.org/indicator/SP.URB.TOTL.IN.ZS
- Health expenditure (% of GDP): https://data.worldbank.org/indicator/SH.XPD.CHEX.GD.ZS

In [2]:
# Request configuration
year = 2024

# Indicators (output name -> World Bank code)

indicators = {
    "population": "SP_POP_TOTL",
    "Surface_area": "AG_LND_TOTL_K2",
    "gdp_per_capita_ppp_current": "NY_GDP_PCAP_PP_CD",
    "gdp_per_capita_usd_current": "NY_GDP_PCAP_CD",
    "life_expectancy": "SP_DYN_LE00_IN",
    "co2_emissions_per_capita": "EN_GHG_CO2_PC_CE_AR5",
    "pm2_5_exposure": "EN_ATM_PM25_MC_M3",
    'urban_population_percentage': "SP_URB_TOTL_IN_ZS",
    'health_expenditure_percentage_of_gdp': "SH_XPD_CHEX_GD_ZS",
}

database_id = "WB_WDI"

In [3]:
indicator_test = dict(list(indicators.items())[:1])
print(indicator_test)

{'population': 'SP_POP_TOTL'}


In [ ]:
def get_page(url, skip):

    try:
        response = requests.get(url,timeout=10, params={"skip" : skip})
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print("HTTP error occurred:", e)
    except requests.exceptions.RequestException as e:
        print("A request error occurred:", e)
        
    if response.status_code != 200:
            print(f"Error while retrieving data: {response.status_code}")
            
    return response.json()

In [ ]:
def get_page(url, skip):
    response = requests.get(url, timeout=10, params={"skip": skip})
    response.raise_for_status()
    return response.json()

In [ ]:
def get_indicator_data(url, name, pagination_limit):
    # first page data
    data = get_page(url, skip=0)
    n = math.ceil(data["count"]/pagination_limit)
    logging.info(f"Number of pages for {name} : {n}")
    
    # other pages data
    for i in range(1,n):
        skip = i * pagination_limit
        data_page =get_page(url, skip=skip)
        data['value'].extend(data_page['value'])

    # catch the case where the number of items returned does not match the expected count
    if len(data['value']) != data['count']:
        raise ValueError(f"Expected {data['count']} items, but got {len(data['value'])}")

    # Transform into a DataFrame and select the desired columns
    df=pd.DataFrame(data['value'])
    nb_duplicates = df.duplicated().sum()
    if nb_duplicates != 0:
        raise ValueError(f"Duplicates found for indicator {name}: {nb_duplicates}")

    return df

In [ ]:
start_year = 2000
end_year = 2024

pagination_limit = 1000  # Pagination limit for requests
for name, code in indicators.items():

    url = f"https://data360api.worldbank.org/data360/data?DATABASE_ID={database_id}&INDICATOR={database_id}_{code}&timePeriodFrom={start_year}&timePeriodTo={end_year}"

    try:
        df = get_indicator_data(url, name, pagination_limit)
        df.to_csv(Path(f"../data/raw/{name}_{start_year}-{end_year}_raw.csv"), index=False)
    
    except (requests.exceptions.RequestException, ValueError) as e:
        logging.error(f"Error for {name}: {e}")
        continue 
     